# L2G-Net from a Cauchy factorization

End-to-end walkthrough: compute the Cauchy factorization of the Minesweeper
graph with [`cauchy_factorization/`](../cauchy_factorization/), export the
spectral tensors, and train L2G-Net on them.

**Paper [ICML-2026 Spotlight]:** [L2G-Net: Local to Global GNNs via Cauchy Factorizations](https://arxiv.org/abs/2602.18837)


## Requirements

```
pip install numpy scipy networkx numba threadpoolctl pymetis   # factorization
pip install torch dgl pyyaml tqdm scikit-learn matplotlib      # training (CUDA)
```


## 1. Dataset

Minesweeper (n = 10000, m = 39402) from the heterophilous benchmark suite
(Platonov et al., 2023).


In [ ]:
!mkdir -p data/minesweeper/raw
!wget -nc -O data/minesweeper/raw/minesweeper.npz \
    https://github.com/yandex-research/heterophilous-graphs/raw/main/data/minesweeper.npz


## 2. Cauchy factorization

Depth 1 (two subgraphs), normalized Laplacian, cut sparsified to a single
bridge edge (`--target-cut 1`, as in the paper). The export also
materializes `basis_cauchy`, the transition matrix between the block-local
and global spectral bases, and probe-checks it against the factorization's
exact transform.

Expected: factorization ~20 s, materialization ~20 s, validation ~1e-15.


In [ ]:
!python export_factorization.py \
    --npz data/minesweeper/raw/minesweeper.npz \
    --target-cut 1 --laplacian norm \
    --out minesweeper_k1.npz --validate


## 3. Inspect the export


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load('minesweeper_k1.npz')
print({k: getattr(d[k], 'shape', d[k]) for k in d.files})

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(np.sort(d['eigvals']), lw=1)
ax[0].set_title('global spectrum (effective Laplacian)')
ax[1].plot(np.sort(d['eig_S']), lw=1, label='S')
ax[1].plot(np.sort(d['eig_T']), lw=1, label='T')
ax[1].set_title('local spectra')
ax[1].legend()
plt.tight_layout()


## 4. Train

One data split as a demo; drop `--num_runs`/`--num_steps` for the full
protocol (10 splits x 1500 steps, ~9 min per split on a GTX 1080 Ti).


In [ ]:
!python train.py --dataset minesweeper --factorization minesweeper_k1.npz \
    --model SGWT --device cuda:0 --num_runs 1 --num_steps 300 --verbose
